# Final analysis — cross-linguistic LLM tokenization

This notebook reloads every CSV produced by the five experiments and
regenerates the major figures. It does **not** re-run any experiment;
the experiments are CLI-driven (see the README).

Run order:
1. `python -m src.run_tokenization_audit --config configs/audit.yaml`
2. `python -m src.run_fixed_context --config configs/fixed_context.yaml`
3. `python -m src.run_radical_sensitivity --config configs/radicals.yaml`
4. `python -m src.run_paraphrase_perturb --config configs/perturb.yaml`
5. `python -m src.run_tokenizer_adaptation --config configs/adaptation.yaml`
6. open this notebook.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from src.plotting import apply_style
from src.experiments.tokenization_audit import (
    plot_mean_tp_bar, plot_tp_box, plot_en_zh_scatter,
)
from src.experiments.fixed_context import (
    _plot_budget_curve, _plot_examples_fit, _plot_cost_per_correct,
)
from src.experiments.tokenizer_adaptation import _plot_before_after

apply_style()
RESULTS = Path('../results')

def maybe(path):
    p = Path(path)
    return pd.read_csv(p) if p.exists() and p.stat().st_size else None

## 1. Tokenization audit

In [ ]:
audit_summary = maybe(RESULTS / 'tokenization_audit' / 'summary_by_tokenizer.csv')
audit_per_sent = maybe(RESULTS / 'tokenization_audit' / 'per_sentence.csv')
if audit_summary is not None:
    display(audit_summary)
    plot_mean_tp_bar(audit_summary, ('en', 'zh'),
                     RESULTS / 'tokenization_audit' / 'plots' / 'tp_bar')
    plot_tp_box(audit_per_sent, ('en', 'zh'),
                RESULTS / 'tokenization_audit' / 'plots' / 'tp_box')
    plot_en_zh_scatter(audit_per_sent, ('en', 'zh'),
                       RESULTS / 'tokenization_audit' / 'plots' / 'en_zh_scatter')
    print('TP > 1 means Chinese fragments more under that tokenizer.\n'
          'Bootstrap CI columns quantify uncertainty over the bilingual sample.')
else:
    print('Run Experiment 1 first.')

## 2. Fixed-context fairness

In [ ]:
fc = maybe(RESULTS / 'fixed_context' / 'summary.csv')
if fc is not None and not fc.empty:
    display(fc)
    base = RESULTS / 'fixed_context' / 'plots'
    _plot_budget_curve(fc, base / 'accuracy_vs_budget', 'accuracy')
    _plot_examples_fit(fc, base / 'examples_fit_vs_budget')
    _plot_cost_per_correct(fc, base / 'cost_per_correct')
    print('examples_fit_mean shows context economics; accuracy combines that with model behaviour.')
else:
    print('Run Experiment 2 first.')

## 3. Radical sensitivity

In [ ]:
rad = maybe(RESULTS / 'radical_sensitivity' / 'condition_table.csv')
coef = maybe(RESULTS / 'radical_sensitivity' / 'logit_coefficients.csv')
if rad is not None: display(rad)
if coef is not None: display(coef)
if rad is None and coef is None:
    print('Run Experiment 3 first.')

## 4. Paraphrase vs token perturbation

In [ ]:
pt = maybe(RESULTS / 'paraphrase_token_perturb' / 'summary.csv')
if pt is not None: display(pt)
else: print('Run Experiment 4 first.')

## 5. Tokenizer adaptation

In [ ]:
ad = maybe(RESULTS / 'tokenizer_adaptation' / 'audit_before_after.csv')
if ad is not None and not ad.empty:
    display(ad)
    _plot_before_after(ad.to_dict(orient='records'),
                       RESULTS / 'tokenizer_adaptation' / 'plots' / 'audit_before_after')
else:
    print('Run Experiment 5 first.')

## Limitations & guardrails

- Numbers are tokenizer-, model-, and task-specific. Generalisations beyond these
  axes are not supported by the data here.
- Closed-model results depend on (unknown) pre-training mixtures.
- The radical sensitivity dataset shipped here is a small seed list.
  Results scale with vocabulary; supply a richer CSV via
  `radicals.yaml: dataset_csv:`.
- Lightweight tokenizer-adaptation results are *counterfactual* unless
  `do_train: true` was set; they upper-bound the gain at zero training cost.